In [22]:
import pandas as pd
seq_regions = ['SEQ_H1', 'SEQ_H2', 'SEQ_L1', 'SEQ_L2', 'SEQ_L3']
cf_regions = ['CF_H1', 'CF_H2', 'CF_L1', 'CF_L2', 'CF_L3']
df = (
    pd.read_csv("data/ab_ag_scalop.tsv", sep="\t")
    .dropna(subset=seq_regions)
    .dropna(subset=cf_regions)
    .drop_duplicates(subset=seq_regions) 
)
antigen_counts = df["antigen_name"].value_counts() # Tabelle aus antigen_names und ihren Häufigkeiten in der Spalte antigen_name
df = df[df["antigen_name"].isin(antigen_counts[antigen_counts >= 5].index)] # Behält nur Zeilen, deren antigen_name mindestens 5-mal vorkommt


In [25]:
import pandas as pd
import os

# CDR-Regionen definieren
cdrs = ["H1", "H2", "L1", "L2", "L3"]

# Mapping zwischen CF-Spalten und neuen HC-Spalten
cf_columns = [f"CF_{cdr}" for cdr in cdrs]
hc_columns = [f"HC_{cdr}" for cdr in cdrs]



# Lege leere HC-Spalten an
for hc_col in hc_columns:
    df[hc_col] = None

# Füge pro CDR-Typ die Clusterlabels hinzu
for cdr in cdrs:
    #cluster_df = pd.read_csv(f"data/cdr_cluster_tsvs/clusters_SEQ_{cdr}.tsv", sep="\t")
    cluster_df = pd.read_csv(f"data/cdr_cluster_tsvs/clusters_SEQ_{cdr}.tsv", sep="\t")
   # Schlüssel erzeugen: Tupel aus (pdb, Hchain, Lchain)
    cluster_df["key"] = list(zip(cluster_df["pdb"], cluster_df["Hchain"], cluster_df["Lchain"]))
    cluster_map = dict(zip(cluster_df["key"], cluster_df["Cluster_Label"]))
    
    # Erzeuge dieselben Schlüssel im Original-DataFrame
    df["key"] = list(zip(df["pdb"], df["Hchain"], df["Lchain"]))
    
    # Neue Spalte füllen
    df[f"HC_{cdr}"] = df["key"].map(cluster_map)

# Schlüssel-Spalte wieder entfernen (optional)
df = df.drop(columns=["key"])

# Speichern als neue Datei
df.to_csv("data/ab_ag_hierarchical_canonical_forms.csv", index=False)



In [26]:
# v measure ohne länge
import pandas as pd
from sklearn.metrics import v_measure_score

# Dateien einlesen
df_true = df  # Original
df_pred = pd.read_csv("data/ab_ag_hierarchical_canonical_forms.csv")  # Hierarchische Cluster

# CDRs, die verglichen werden sollen
cdrs = ["H1", "H2", "L1", "L2", "L3"]

print("V-Measure Ergebnisse:\n")

# Für jede Region CF vs HC vergleichen
for cdr in cdrs:
    true_labels = df_true[f"CF_{cdr}"]
    pred_labels = df_pred[f"HC_{cdr}"]
    
    v_score = v_measure_score(true_labels, pred_labels)
    print(f"CDR {cdr}: V-Measure = {v_score:.3f}")

V-Measure Ergebnisse:

CDR H1: V-Measure = 0.236
CDR H2: V-Measure = 0.306
CDR L1: V-Measure = 0.418
CDR L2: V-Measure = 0.000
CDR L3: V-Measure = 0.416


In [19]:
import pandas as pd


# CDR-Regionen definieren
cdrs = ["H1", "H2", "L1", "L2", "L3"]

# Mapping zwischen CF-Spalten und neuen HC-Spalten
cf_columns = [f"CF_{cdr}" for cdr in cdrs]
hc_columns = [f"HC_{cdr}" for cdr in cdrs]

# Lege neue HC-Spalten mit leeren Werten an
for hc_col in hc_columns:
    df[hc_col] = None

# Für jede CDR-Typ die passende Clusterdatei einlesen und zuordnen
for cdr in cdrs:
    cluster_df = pd.read_csv(f"data/merged_clusters_{cdr}.csv")

   # Schlüssel erzeugen: Tupel aus (pdb, Hchain, Lchain)
    cluster_df["key"] = list(zip(cluster_df["pdb"], cluster_df["Hchain"], cluster_df["Lchain"]))
    cluster_map = dict(zip(cluster_df["key"], cluster_df["cluster"]))
    
    # Erzeuge dieselben Schlüssel im Original-DataFrame
    df["key"] = list(zip(df["pdb"], df["Hchain"], df["Lchain"]))
    
    # Neue Spalte füllen
    df[f"HC_{cdr}"] = df["key"].map(cluster_map)

# Schlüssel-Spalte wieder entfernen (optional)
df = df.drop(columns=["key"])
# Speichern als neue Datei
df.to_csv("data/ab_ag_hierarchical_canonical_forms_nach_länge.csv", index=False)


In [20]:
import pandas as pd
from sklearn.metrics import v_measure_score

# Dateien einlesen
df_true =df  # Original
df_pred = pd.read_csv("data/ab_ag_hierarchical_canonical_forms_nach_länge.csv")  # neue Datei mit Länge-Clustern

# CDRs, die verglichen werden sollen
cdrs = ["H1", "H2", "L1", "L2", "L3"]

print("V-Measure Ergebnisse:\n")

# Für jede Region CF vs HC vergleichen
for cdr in cdrs:
    true_labels = df_true[f"CF_{cdr}"]
    pred_labels = df_pred[f"HC_{cdr}"]
    
    v_score = v_measure_score(true_labels, pred_labels)
    print(f"CDR {cdr}: V-Measure = {v_score:.3f}")



V-Measure Ergebnisse:

CDR H1: V-Measure = 0.371
CDR H2: V-Measure = 0.532
CDR L1: V-Measure = 0.740
CDR L2: V-Measure = 0.000
CDR L3: V-Measure = 0.545


In [16]:
#versuch die zuordnung über pdb hchain und lchain zu machen
import pandas as pd


# CDR-Regionen definieren
cdrs = ["H1", "H2", "L1", "L2", "L3"]

# Mapping zwischen CF-Spalten und neuen HC-Spalten
cf_columns = [f"CF_{cdr}" for cdr in cdrs]
hc_columns = [f"HC_{cdr}" for cdr in cdrs]

# Lege neue HC-Spalten mit leeren Werten an
for hc_col in hc_columns:
    df[hc_col] = None

# Für jede CDR-Typ die passende Clusterdatei einlesen und zuordnen
for cdr in cdrs:
    cluster_df = pd.read_csv(f"data/merged_clusters_pre_decided_{cdr}.csv")

    # Schlüssel erzeugen: Tupel aus (pdb, Hchain, Lchain)
    cluster_df["key"] = list(zip(cluster_df["PDB_ID"], cluster_df["Hchain"], cluster_df["Lchain"]))
    cluster_map = dict(zip(cluster_df["key"], cluster_df["cluster"]))
    
    # Erzeuge dieselben Schlüssel im Original-DataFrame
    df["key"] = list(zip(df["pdb"], df["Hchain"], df["Lchain"]))
    
    # Neue Spalte füllen
    df[f"HC_{cdr}"] = df["key"].map(cluster_map)

# Schlüssel-Spalte wieder entfernen (optional)
df = df.drop(columns=["key"])

# Speichern als neue Datei
df.to_csv("data/ab_ag_hierarchical_canonical_forms_pre_decided_length.csv", index=False)


In [18]:
#mit pre decided length
# Dateien einlesen
df_true =df  # Original
df_pred = pd.read_csv("data/ab_ag_hierarchical_canonical_forms_pre_decided_length.csv")  # neue Datei mit Länge-Clustern

# CDRs, die verglichen werden sollen
cdrs = ["H1", "H2", "L1", "L2", "L3"]

print("V-Measure Ergebnisse:\n")

# Für jede Region CF vs HC vergleichen
for cdr in cdrs:
    true_labels = df_true[f"CF_{cdr}"]
    pred_labels = df_pred[f"HC_{cdr}"]
    
    v_score = v_measure_score(true_labels, pred_labels)
    print(f"CDR {cdr}: V-Measure = {v_score:.3f}")

V-Measure Ergebnisse:

CDR H1: V-Measure = 0.357
CDR H2: V-Measure = 0.579
CDR L1: V-Measure = 0.778
CDR L2: V-Measure = 1.000
CDR L3: V-Measure = 0.731
